# 6. Interactive Map of the Launch Sites (Folium)

**Goal:** see where the launch sites are, how successful each one is, and what is near them. GitHub cannot show interactive maps inside a notebook, so each map is also saved to the `maps/` folder. Download an HTML file and open it in a browser to use it.

In [1]:
from math import atan2, cos, radians, sin, sqrt

import folium
import pandas as pd
from folium.features import DivIcon
from folium.plugins import MarkerCluster, MousePosition

spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
spacex_df = spacex_df[["Launch Site", "Lat", "Long", "class"]]
launch_sites_df = spacex_df.groupby(["Launch Site"], as_index=False).first()[["Launch Site", "Lat", "Long"]]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


## Task 1: Mark all launch sites on a map

In [2]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, site in launch_sites_df.iterrows():
    coordinate = [site["Lat"], site["Long"]]
    folium.Circle(coordinate, radius=1000, color="#d35400", fill=True).add_child(
        folium.Popup(site["Launch Site"])).add_to(site_map)
    folium.Marker(coordinate, icon=DivIcon(
        icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % site["Launch Site"],
    )).add_to(site_map)

site_map.save("../maps/1_launch_sites.html")
site_map

All four sites are on a coast, so rockets fly out over the ocean and a failed launch falls into the sea. The three Florida sites are as far south as the US mainland allows, and a site closer to the equator gets more speed from the Earth's rotation.

## Task 2: Mark the successful and failed launches at each site

Green markers landed and red markers did not. Markers at the same site are grouped into a cluster until you zoom in.

In [3]:
spacex_df["marker_color"] = spacex_df["class"].map({1: "green", 0: "red"})

site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster().add_to(site_map)
for _, record in spacex_df.iterrows():
    folium.Marker([record["Lat"], record["Long"]],
                  icon=folium.Icon(color="white", icon_color=record["marker_color"])).add_to(marker_cluster)

site_map.save("../maps/2_launch_outcomes.html")
site_map

In [4]:
# Success rate and number of launches at each site
spacex_df.groupby("Launch Site")["class"].agg(launches="count", success_rate="mean").round(2)

,launches,success_rate
Launch Site,,
CCAFS LC-40,26,0.27
CCAFS SLC-40,7,0.43
KSC LC-39A,13,0.77
VAFB SLC-4E,10,0.40


## Task 3: Distances from a launch site to its surroundings

The distances below are from CCAFS SLC-40 to the nearest coastline and to two nearby cities. The coastline point was read from the map with the mouse-position tool. The city points are the city centers.

In [5]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # approximate radius of the Earth in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


site = launch_sites_df.set_index("Launch Site").loc["CCAFS SLC-40"]
site_coordinate = [site["Lat"], site["Long"]]
places = {"Coastline": [28.56367, -80.57163],
          "Titusville": [28.6122, -80.8076],
          "Melbourne": [28.0836, -80.6081]}

site_map = folium.Map(location=site_coordinate, zoom_start=10)
site_map.add_child(MousePosition(position="topright", separator=" Long: ", prefix="Lat:"))
folium.Marker(site_coordinate, popup="CCAFS SLC-40").add_to(site_map)

distances = {}
for name, coordinate in places.items():
    distance = calculate_distance(*site_coordinate, *coordinate)
    distances[name] = round(distance, 2)
    folium.Marker(coordinate, icon=DivIcon(
        icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s: %.2f km</b></div>' % (name, distance),
    )).add_to(site_map)
    folium.PolyLine(locations=[site_coordinate, coordinate], weight=1).add_to(site_map)

site_map.save("../maps/3_proximities.html")
print(distances)
site_map

{'Coastline': 0.51, 'Titusville': 23.19, 'Melbourne': 53.43}


The launch site is about half a kilometer from the coast but 23 to 53 km from the nearest cities, so launches can fly over water and away from people.